In [0]:
# =========================================================
# PROJECT GOLD LAYER PRODUCTION TESTS (FIXED)
# Database: vstone_catalog.gold
# =========================================================
from pyspark.sql import functions as F

CATALOG = "vstone_catalog"
GOLD = f"{CATALOG}.gold"

def test_gold_fact_table_populated():
    """Requirement: Gold fact_listings must have significant rows"""
    cnt = spark.table(f"{GOLD}.fact_listings").count()
    # Fact listings typically has ~1.1M rows
    assert cnt > 100000, f"FAIL: fact_listings has only {cnt} rows — too few for the source volume!"
    print(f"✅ PASS: fact_listings — {cnt:,} rows")

def test_gold_scd2_columns_on_all_dims():
    """Requirement: All dimensions must have SCD2 columns (Upper Case)"""
    # Names mapped directly from your Gold catalog
    scd2_dims = ["dim_car", "dim_location", "dim_listing_details", "dim_date"]
    for dim in scd2_dims:
        df = spark.table(f"{GOLD}.{dim}")
        # DLT Metadata columns check
        assert "__START_AT" in df.columns, f"FAIL: {dim} missing __START_AT!"
        assert "__END_AT" in df.columns, f"FAIL: {dim} missing __END_AT!"
        print(f"✅ PASS: {dim} has SCD2 columns (__START_AT, __END_AT)")

def test_gold_all_aggregates_populated():
    """Requirement: Verify all 4 Aggregate tables from Catalog"""
    # Updated to match EXACT names in your 'gold' catalog image
    agg_tables = [
        "agg_brand_location_performance",
        "agg_comprehensive_kpi_cube",
        "agg_monthly_sales_trend",
        "agg_regional_market_depth"
    ]
    for agg in agg_tables:
        cnt = spark.table(f"{GOLD}.{agg}").count()
        assert cnt > 0, f"FAIL: Aggregate {agg} is EMPTY!"
        print(f"✅ PASS: {agg} — {cnt:,} rows")

def test_gold_kpi_requirements():
    """Requirement: Check specific KPIs (Monthly Trends & Top Brands)"""
    # 1. Monthly sales trend check
    trend_cnt = spark.table(f"{GOLD}.agg_monthly_sales_trend").count()
    assert trend_cnt >= 12, f"FAIL: Not enough data points for Monthly Trend!"
    
    # 2. Regional depth check (Sales by region/category)
    region_cnt = spark.table(f"{GOLD}.agg_regional_market_depth").count()
    assert region_cnt > 0, "FAIL: Regional market analysis is empty!"
    print(f"✅ PASS: Monthly and Regional KPI requirements met.")

def test_gold_geolocation_join_quality():
    """Requirement: Verify join rate between Fact and Location Dimension"""
    fact_cnt = spark.table(f"{GOLD}.fact_listings").count()
    # Using specific keys city_prepositional as PK
    joined_cnt = spark.sql(f"""
        SELECT COUNT(*) as c
        FROM {GOLD}.fact_listings f
        JOIN {GOLD}.dim_location d ON f.location_key = d.city_prepositional
        WHERE d.__END_AT IS NULL
    """).collect()[0]['c']
    
    join_rate = round(joined_cnt / fact_cnt * 100, 1)
    # 60% threshold requirement
    assert join_rate >= 60.0, f"FAIL: Low join rate {join_rate}% (Min 60% required)!"
    print(f"✅ PASS: Geolocation join rate = {join_rate}% ({joined_cnt:,}/{fact_cnt:,} listings)")

# --- EXECUTION ---
print("🚀 Starting Project Gold Layer Tests...\n")
try:
    test_gold_fact_table_populated()
    test_gold_scd2_columns_on_all_dims()
    test_gold_all_aggregates_populated()
    test_gold_kpi_requirements()
    test_gold_geolocation_join_quality()
    print("\n🎯 ALL GOLD TESTS PASSED!")
except AssertionError as e:
    print(f"\n🛑 TEST FAILED: {str(e)}")
except Exception as e:
    print(f"\n💥 SYSTEM ERROR: {str(e)}")